In [ ]:
#1: Khai báo thư viện và tải dữ liệu FashionMNIST
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD, Adam
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
%matplotlib inline

# Cấu hình thiết bị chạy (Tự động nhận diện GPU/CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_folder = './data/FMNIST' 
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True) 
tr_images = fmnist.data 
tr_targets = fmnist.targets 
 
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False) 
val_images = val_fmnist.data 
val_targets = val_fmnist.targets 

# Kiểm tra các kích thước tensor
unique_values = tr_targets.unique() 
print(f'tr_images & tr_targets:\n\tX - {tr_images.shape}\n\tY - {tr_targets.shape}\n\tY - Unique Values : {unique_values}') 
print(f'TASK:\n\t{len(unique_values)} class Classification') 
print(f'UNIQUE CLASSES:\n\t{fmnist.classes}')

In [ ]:
#2: Vẽ lưới mẫu ảnh 10 x 10
R, C = len(tr_targets.unique()), 10 
fig, ax = plt.subplots(R, C, figsize=(10,10)) 
for label_class, plot_row in enumerate(ax): 
    label_x_rows = np.where(tr_targets == label_class)[0] 
    for plot_cell in plot_row: 
        plot_cell.grid(False)
        plot_cell.axis('off') 
        ix = np.random.choice(label_x_rows) 
        x, y = tr_images[ix], tr_targets[ix] 
        plot_cell.imshow(x, cmap='gray') 
plt.tight_layout()
plt.show()

In [ ]:
#3: Khởi tạo Lớp Dataset có chia tỷ lệ (x / 255) và các hàm bổ trợ
# Định nghĩa Dataset có Scale dữ liệu
class FMNISTDataset(Dataset): 
    def __init__(self, x, y): 
        x = x.float() / 255 
        x = x.view(-1, 28 * 28) 
        self.x, self.y = x, y  
    def __getitem__(self, ix): 
        return self.x[ix].to(device), self.y[ix].to(device) 
    def __len__(self):  
        return len(self.x) 

def train_batch(x, y, model, opt, loss_fn): 
    model.train() 
    prediction = model(x) 
    batch_loss = loss_fn(prediction, y) 
    batch_loss.backward() 
    opt.step()             # Đổi thành opt để linh hoạt theo đối tượng truyền vào
    opt.zero_grad() 
    return batch_loss.item() 
 
def accuracy(x, y, model): 
    model.eval() 
    with torch.no_grad(): 
        prediction = model(x) 
    _, argmaxes = prediction.max(-1) 
    is_correct = argmaxes == y 
    return is_correct.cpu().numpy().tolist() 

def get_data():  
    train = FMNISTDataset(tr_images, tr_targets)  
    trn_dl = DataLoader(train, batch_size=32, shuffle=True) 
    val = FMNISTDataset(val_images, val_targets)  
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False) 
    return trn_dl, val_dl 
 
@torch.no_grad() 
def val_loss(x, y, model): 
    prediction = model(x) 
    val_loss = loss_fn(prediction, y) 
    return val_loss.item()

In [ ]:
#4: Thí nghiệm 1 – Tốc độ học CAO (lr = 0.1) trên dữ liệu ĐÃ chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-1) # lr = 0.1
    return model, loss_fn, optimizer 
 
trn_dl, val_dl = get_data() 
model, loss_fn, optimizer = get_model() 
 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}") 
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)    
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 

# Trực quan hóa
epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.1 learning rate') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.1 learning rate') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])  
plt.legend(); plt.grid(False) 
plt.show()

In [ ]:
#5: Thí nghiệm 2 – Tốc độ học TRUNG BÌNH (lr = 0.001) trên dữ liệu ĐÃ chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-3) # lr = 0.001
    return model, loss_fn, optimizer 

trn_dl, val_dl = get_data() 
model, loss_fn, optimizer = get_model() 
 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}") 
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)):    
        x, y = batch 
        batch_loss = train_batch(x, y, model, optimizer, loss_fn) 
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 
 
epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.001 learning rate') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.001 learning rate') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()]) 
plt.legend(); plt.grid(False) 
plt.show()

In [ ]:
#6: Thí nghiệm 3 – Tốc độ học THẤP (lr = 0.00001) trên dữ liệu ĐÃ chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-5) # lr = 0.00001
    return model, loss_fn, optimizer  
 
trn_dl, val_dl = get_data() 
model, loss_fn, optimizer = get_model() 
 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}") 
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch  
        batch_loss = train_batch(x, y, model, optimizer, loss_fn) 
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 

epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.00001 learning rate') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.00001 learning rate') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])  
plt.legend(); plt.grid(False) 
plt.show()

In [ ]:
#7: Khởi tạo lại Lớp Dataset KHÔNG chia tỷ lệ (Giữ nguyên gốc x.float())
# Định nghĩa Dataset KHÔNG Scale (Không chia 255)
class FMNISTDatasetUnscaled(Dataset): 
    def __init__(self, x, y): 
        x = x.float()            # Giữ nguyên giá trị thô [0, 255]
        x = x.view(-1, 28 * 28) 
        self.x, self.y = x, y  
    def __getitem__(self, ix): 
        return self.x[ix].to(device), self.y[ix].to(device) 
    def __len__(self):  
        return len(self.x) 

def get_data_unscaled():  
    train = FMNISTDatasetUnscaled(tr_images, tr_targets)  
    trn_dl = DataLoader(train, batch_size=32, shuffle=True) 
    val = FMNISTDatasetUnscaled(val_images, val_targets)  
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False) 
    return trn_dl, val_dl

In [ ]:
#8: Thí nghiệm 4 – Tốc độ học CAO (lr = 0.1) trên dữ liệu KHÔNG chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-1) 
    return model, loss_fn, optimizer 
 
trn_dl, val_dl = get_data_unscaled() 
model, loss_fn, optimizer = get_model() 
 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}") 
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        batch_loss = train_batch(x, y, model, optimizer, loss_fn) 
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 
 
epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.1 learning rate (Unscaled)') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.1 learning rate (Unscaled)') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])  
plt.legend(); plt.grid(False) 
plt.show()

In [ ]:
#9: Thí nghiệm 5 – Tốc độ học TRUNG BÌNH (lr = 0.001) trên dữ liệu KHÔNG chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-3) 
    return model, loss_fn, optimizer 

trn_dl, val_dl = get_data_unscaled() 
model, loss_fn, optimizer = get_model() 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}") 
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch  
        batch_loss = train_batch(x, y, model, optimizer, loss_fn) 
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 

epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.001 learning rate (Unscaled)') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.001 learning rate (Unscaled)') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])  
plt.legend(); plt.grid(False) 
plt.show()

In [ ]:
#10: Thí nghiệm 6 – Tốc độ học THẤP (lr = 0.00001) trên dữ liệu KHÔNG chia tỷ lệ
def get_model(): 
    model = nn.Sequential( 
        nn.Linear(28 * 28, 1000), 
        nn.ReLU(), 
        nn.Linear(1000, 10) 
    ).to(device) 
    loss_fn = nn.CrossEntropyLoss() 
    optimizer = Adam(model.parameters(), lr=1e-5) 
    return model, loss_fn, optimizer  

trn_dl, val_dl = get_data_unscaled() 
model, loss_fn, optimizer = get_model() 
 
train_losses, train_accuracies = [], [] 
val_losses, val_accuracies = [], [] 
for epoch in range(5): 
    print(f"Epoch: {epoch}")   
    train_epoch_losses, train_epoch_accuracies = [], [] 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        batch_loss = train_batch(x, y, model, optimizer, loss_fn) 
        train_epoch_losses.append(batch_loss)  
    train_epoch_loss = np.array(train_epoch_losses).mean() 
 
    for ix, batch in enumerate(iter(trn_dl)): 
        x, y = batch 
        is_correct = accuracy(x, y, model) 
        train_epoch_accuracies.extend(is_correct) 
    train_epoch_accuracy = np.mean(train_epoch_accuracies) 
    for ix, batch in enumerate(iter(val_dl)): 
        x, y = batch 
        val_is_correct = accuracy(x, y, model) 
        validation_loss = val_loss(x, y, model) 
    val_epoch_accuracy = np.mean(val_is_correct) 
    train_losses.append(train_epoch_loss) 
    train_accuracies.append(train_epoch_accuracy) 
    val_losses.append(validation_loss) 
    val_accuracies.append(val_epoch_accuracy) 

epochs = np.arange(5) + 1 
plt.figure(figsize=(10, 8))
plt.subplot(211) 
plt.plot(epochs, train_losses, 'bo', label='Training loss') 
plt.plot(epochs, val_losses, 'r', label='Validation loss') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation loss with 0.00001 learning rate (Unscaled)') 
plt.ylabel('Loss'); plt.legend(); plt.grid(False) 
plt.subplot(212) 
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy') 
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy') 
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1)) 
plt.title('Training and validation accuracy with 0.00001 learning rate (Unscaled)') 
plt.xlabel('Epochs'); plt.ylabel('Accuracy') 
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])  
plt.legend(); plt.grid(False) 
plt.show()